# NB11x_G: Consolidate NB11 Tuning Results into the Result Registry

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Reads the NB11 hyperparameter-tuning artifacts (`best_params/*.json`, `summary/*.csv`, `oof/*.parquet`) and writes correct experiment records into `results/registry` under the dedicated label `NB11_V2`, with `registry.save()`. This is the root-cause fix for the missing tuned results: NB11 logged in-memory under `notebook='NB08c_v2'` and never saved. After running this once, the registry-driven notebooks (NB13d report, NB13e tuning comparison) pick up the 31 tuned studies automatically via `ResultRegistry.load_all`. The OOF-driven notebooks (NB13a/b/c) already read the OOF parquets directly.

Per study it logs two rows sharing `cell_id = study_tag`: a **tuned** row (`<clf>-Optuna`, full metrics + per-city + fold metrics computed from OOF) and a **baseline** row (`<clf>` stem, baseline AUC only) so NB13e's `find_baseline_row` resolves the pair.

## CELL 1 -- CONFIG + GLOBAL SETUP

In [1]:
# @title CELL 1: NB11x_G CONFIG + GLOBAL SETUP
import platform, os
if platform.system() == "Windows":
    _setup = r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks\global_setup.py"
elif os.path.exists("/content/drive_f"):
    _setup = "/content/drive_f/masterthesis/notebooks/global_setup.py"
else:
    _setup = "/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks/global_setup.py"
with open(_setup) as f:
    exec(f.read())

import sys, json as _json
from pathlib import Path
from datetime import datetime as _dt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
from bda_results import ResultRegistry

NB11_V2_DIR      = OUTPUTS_DIR / 'NB11_V2'
NB11_BEST_PARAMS = NB11_V2_DIR / 'best_params'
NB11_SUMMARY_DIR = NB11_V2_DIR / 'summary'
NB11_OOF_DIR     = NB11_V2_DIR / 'oof'
REGISTRY_DIR     = RESULTS_ROOT / 'registry'

# correct registry label. NB11 originally logged under 'NB08c_v2' which would
# clobber the real NB08c rows; here we use a dedicated label so the NB13 series
# reads NB11 results via ResultRegistry.load_all without per-notebook backfills.
REGISTRY_NOTEBOOK = 'NB11_V2'

print("=" * 70)
print("NB11x_G: CONSOLIDATE NB11 TUNING RESULTS INTO REGISTRY")
print("=" * 70)
print(f"  best_params: {NB11_BEST_PARAMS}  exists={NB11_BEST_PARAMS.exists()}")
print(f"  summary:     {NB11_SUMMARY_DIR}  exists={NB11_SUMMARY_DIR.exists()}")
print(f"  oof:         {NB11_OOF_DIR}  exists={NB11_OOF_DIR.exists()}")
print(f"  registry:    {REGISTRY_DIR}")
print(f"  registry notebook label: {REGISTRY_NOTEBOOK}")


BDA GLOBAL SETUP
Started: 2026-07-03 08:33:53
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------


/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


<string>:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     930.7/7452.0 GB (6521.3 GB free)
  GDrive (F:)     1405.7/3726.0 GB (2320.3 GB free)
  Local data      11563.4/14901.9 GB (3338.4 GB free)
  Data stack      1405.7/3726.0 GB (2320.3 GB free)
  WSL ext4        69.1/1006.9 GB (886.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

## CELL S0 -- LOAD NB11 ARTIFACTS (best_params + summary)

In [2]:
# @title CELL S0: LOAD NB11 ARTIFACTS (best_params + summary)
print("=" * 70)
print("CELL S0: LOAD NB11 ARTIFACTS")
print("=" * 70)

# best_params JSONs -- authoritative per-study source (has fold_aucs + tuned params)
BP = []
for fp in sorted(NB11_BEST_PARAMS.glob('*.json')):
    with open(fp) as f:
        d = _json.load(f)
    d['_path'] = str(fp)
    BP.append(d)
print(f"  best_params studies: {len(BP)}")

# summary CSVs -- cross-check (same tuned_auc/baseline_auc/delta)
SUM = []
for fp in sorted(NB11_SUMMARY_DIR.glob('*_summary.csv')):
    try:
        SUM.append(pd.read_csv(fp))
    except Exception as e:
        print(f"  WARN read {fp.name}: {e}")
SUM_DF = pd.concat(SUM, ignore_index=True) if SUM else pd.DataFrame()
print(f"  summary rows: {len(SUM_DF)} from {len(SUM)} files")

# normalize best_params into a flat per-study table
STUDIES = []
for d in BP:
    clf = str(d.get('classifier'))
    manifest = str(d.get('manifest_key'))
    STUDIES.append({
        'study_tag': d.get('study_tag'),
        'notebook_src': d.get('notebook'),
        'manifest_key': manifest,
        'parquet_id': d.get('parquet_id'),
        'feature_set': d.get('feature_set'),
        'classifier': clf,
        'tuned_classifier': f"{clf}-Optuna",
        'baseline_classifier': clf,                 # stem -> matches NB13e baseline family
        'parquet_name': f"bda_{manifest}_v2",
        'baseline_auc': d.get('baseline_auc'),
        'tuned_auc': d.get('tuned_auc'),
        'delta_auc': d.get('delta_auc'),
        'mean_folds_auc': d.get('mean_folds_auc'),
        'std_folds_auc': d.get('std_folds_auc'),
        'fold_aucs': d.get('fold_aucs'),
        'n_trials_done': d.get('n_trials_done'),
        'n_trials_requested': d.get('n_trials_requested'),
        'n_features': d.get('n_features'),
        'n_buildings': d.get('n_buildings'),
        'n_cities': d.get('n_cities'),
        'best_params': d.get('best_params'),
    })
STUDIES_DF = pd.DataFrame(STUDIES)
assert len(STUDIES_DF) == len(BP), "study normalization lost rows"
print(f"  studies normalized: {len(STUDIES_DF)}")

# cross-check best_params vs summary tuned_auc (by study_tag)
if len(SUM_DF) > 0 and 'study_tag' in SUM_DF.columns:
    _chk = STUDIES_DF.merge(SUM_DF[['study_tag', 'tuned_auc']], on='study_tag',
                            how='left', suffixes=('', '_sum'))
    _chk['drift'] = (_chk['tuned_auc'] - _chk['tuned_auc_sum']).abs()
    _bad = _chk[_chk['drift'] > 1e-6]
    print(f"  best_params vs summary tuned_auc drift > 1e-6: {len(_bad)}")

print()
print(STUDIES_DF[['study_tag', 'tuned_classifier', 'baseline_auc',
                  'tuned_auc', 'delta_auc', 'mean_folds_auc']].to_string(index=False))


CELL S0: LOAD NB11 ARTIFACTS
  best_params studies: 31
  summary rows: 31 from 14 files
  studies normalized: 31
  best_params vs summary tuned_auc drift > 1e-6: 0

                                                                        study_tag    tuned_classifier  baseline_auc  tuned_auc  delta_auc  mean_folds_auc
                           NB11a_F2_F6__fusion_indices_card_cohdrop__LightGBM-MIA LightGBM-MIA-Optuna      0.815426   0.847121   0.031695        0.748124
                                NB11a_F2_F6__fusion_ms_card_cohdrop__LightGBM-MIA LightGBM-MIA-Optuna      0.813722   0.839066   0.025344        0.728085
                                      NB11a_F3__fusion_card_cohdrop__LightGBM-MIA LightGBM-MIA-Optuna      0.841694   0.845727   0.004032        0.724197
                              NB11a_F4_F7__fusion_composite_cohdrop__LightGBM-MIA LightGBM-MIA-Optuna      0.812965   0.844908   0.031943        0.678253
                                     NB11a_F4_F7__fusion_ms_cohdr

## CELL M1 -- LOG TO REGISTRY (tuned + baseline) AND SAVE

In [3]:
# @title CELL M1: LOG NB11 RESULTS INTO REGISTRY (tuned + baseline) AND SAVE
print("=" * 70)
print("CELL M1: LOG TO REGISTRY")
print("=" * 70)

registry = ResultRegistry(RESULTS_ROOT, notebook=REGISTRY_NOTEBOOK)

n_oof = 0
n_no_oof = 0
for st in STUDIES:
    study_tag = st['study_tag']
    manifest = st['manifest_key']
    model_id = f"{study_tag}-Optuna"
    cell_id = study_tag    # shared key so NB13e find_baseline_row pairs tuned<->baseline

    fold_aucs = st['fold_aucs'] if isinstance(st['fold_aucs'], list) else []
    fold_metrics = {
        'auc_mean': st['mean_folds_auc'],
        'auc_std': st['std_folds_auc'],
        'auc_min': float(np.min(fold_aucs)) if fold_aucs else None,
        'auc_max': float(np.max(fold_aucs)) if fold_aucs else None,
        'fold_aucs': fold_aucs,
        'n_folds': len(fold_aucs) if fold_aucs else None,
    }
    extra = {
        'baseline_auc': st['baseline_auc'],
        'tuned_auc_summary': st['tuned_auc'],
        'delta_auc': st['delta_auc'],
        'mean_folds_auc': st['mean_folds_auc'],
        'std_folds_auc': st['std_folds_auc'],
        'fold_aucs': fold_aucs,
        'n_trials_done': st['n_trials_done'],
        'n_trials_requested': st['n_trials_requested'],
        'best_params': st['best_params'],
        'study_tag': study_tag,
    }
    imp = 'native_nan' if 'MIA' in str(st['classifier']).upper() else 'median'
    tags_tuned = ['nb11_optuna', 'tuned', manifest, st['tuned_classifier']]

    cands = sorted(NB11_OOF_DIR.glob(f"oof_{model_id}__*.parquet"),
                   key=lambda p: p.stat().st_mtime, reverse=True)

    if cands:
        oof = pd.read_parquet(cands[0], columns=['y_true', 'y_proba', 'city', 'fold_id'])
        rec = registry.log_experiment(
            notebook=REGISTRY_NOTEBOOK, cell_id=cell_id,
            experiment_name=study_tag,
            parquet_name=st['parquet_name'],
            feature_set_name=str(st['feature_set']),
            classifier_name=st['tuned_classifier'],
            classifier_params=st['best_params'],
            cv_method='GroupKFold', n_folds=int(oof['fold_id'].nunique()),
            imputation=imp, tier_selection=[0, 1, 2],
            y_true=oof['y_true'].astype(float).to_numpy(),
            y_proba=oof['y_proba'].astype(float).to_numpy(),
            groups=oof['city'].astype(str).to_numpy(),
            fold_metrics=fold_metrics,
            oof_path=str(cands[0]),
            note=(f"NB11 Optuna tuned; pooled_auc={st['tuned_auc']:.4f} "
                  f"mean_folds={st['mean_folds_auc']:.4f} baseline={st['baseline_auc']:.4f} "
                  f"delta={st['delta_auc']:.4f}"),
            tags=tags_tuned, extra=extra,
        )
        n_oof += 1
    else:
        rec = registry.log_experiment(
            notebook=REGISTRY_NOTEBOOK, cell_id=cell_id,
            experiment_name=study_tag,
            parquet_name=st['parquet_name'],
            feature_set_name=str(st['feature_set']),
            classifier_name=st['tuned_classifier'],
            classifier_params=st['best_params'],
            cv_method='GroupKFold', imputation=imp, tier_selection=[0, 1, 2],
            metrics={'auc': st['tuned_auc']},
            fold_metrics=fold_metrics,
            note=f"NB11 Optuna tuned [no OOF found]; pooled_auc={st['tuned_auc']}",
            tags=tags_tuned, extra=extra,
        )
        n_no_oof += 1

    # set true feature count without storing a junk feature_cols list
    rec['n_features'] = int(st['n_features']) if st['n_features'] else 0

    # baseline sibling row (AUC only) so NB13e pairing resolves
    registry.log_experiment(
        notebook=REGISTRY_NOTEBOOK, cell_id=cell_id,
        experiment_name=f"{study_tag}__baseline",
        parquet_name=st['parquet_name'],
        feature_set_name=str(st['feature_set']),
        classifier_name=st['baseline_classifier'],
        cv_method='GroupKFold', imputation=imp, tier_selection=[0, 1, 2],
        n_buildings=st['n_buildings'],
        metrics={'auc': st['baseline_auc']},
        note=f"NB11 source baseline for {study_tag} (pre-tuning AUC from best_params)",
        tags=['nb11_baseline', manifest, st['baseline_classifier']],
    )

print(f"\n  tuned logged from OOF: {n_oof}, summary-only: {n_no_oof}")
print(f"  baseline rows: {len(STUDIES)}")
print(f"  total records in memory: {len(registry.experiments)}")

saved_dir = registry.save()
print(f"  registry saved under notebook label '{REGISTRY_NOTEBOOK}' to: {saved_dir}")


CELL M1: LOG TO REGISTRY
  ResultRegistry: /content/drive_f/masterthesis/results/registry (run_id=20260703_083441)
  REG: NB11a_F2_F6__fusion_indices_card_cohdrop__LightGBM-MIA AUC=0.8471 F1=0.0891 n=466704 feat=0 cities=18 [NB11_V2/NB11a_F2_F6__fusion_indices_card_cohdrop__LightGBM-MIA]
  REG: NB11a_F2_F6__fusion_indices_card_cohdrop__LightGBM-MIA__baseline AUC=0.8154 F1=? n=466704 feat=0 cities=0 [NB11_V2/NB11a_F2_F6__fusion_indices_card_cohdrop__LightGBM-MIA]
  REG: NB11a_F2_F6__fusion_ms_card_cohdrop__LightGBM-MIA AUC=0.8391 F1=0.0901 n=466704 feat=0 cities=18 [NB11_V2/NB11a_F2_F6__fusion_ms_card_cohdrop__LightGBM-MIA]
  REG: NB11a_F2_F6__fusion_ms_card_cohdrop__LightGBM-MIA__baseline AUC=0.8137 F1=? n=466704 feat=0 cities=0 [NB11_V2/NB11a_F2_F6__fusion_ms_card_cohdrop__LightGBM-MIA]
  REG: NB11a_F3__fusion_card_cohdrop__LightGBM-MIA   AUC=0.8457 F1=0.0804 n=584986 feat=0 cities=20 [NB11_V2/NB11a_F3__fusion_card_cohdrop__LightGBM-MIA]
  REG: NB11a_F3__fusion_card_cohdrop__LightGBM-

## CELL M2 -- VERIFY (reload, integrity, AUC cross-check)

In [4]:
# @title CELL M2: VERIFY REGISTRY (reload + checks)
print("=" * 70)
print("CELL M2: VERIFY")
print("=" * 70)

experiments, _profiles = ResultRegistry.load_all(RESULTS_ROOT)
EDF = ResultRegistry.experiments_to_dataframe(experiments)
nb11 = EDF[EDF['notebook'] == REGISTRY_NOTEBOOK].copy()
is_tuned = nb11['classifier_name'].fillna('').str.endswith('-Optuna')
tuned = nb11[is_tuned]
base = nb11[~is_tuned]
print(f"  NB11_V2 rows in registry: {len(nb11)}  (tuned={len(tuned)}, baseline={len(base)})")
assert len(tuned) == len(STUDIES), f"expected {len(STUDIES)} tuned rows, got {len(tuned)}"
assert len(base) == len(STUDIES), f"expected {len(STUDIES)} baseline rows, got {len(base)}"

# NB08c integrity: confirm we did not clobber the real NB08c rows
for _nb in ['NB08c_v2', 'NB09a_v2']:
    _n = int((EDF['notebook'] == _nb).sum())
    print(f"  integrity: {_nb} rows still present = {_n}")

# AUC cross-check: registry pooled (from OOF) vs best_params tuned_auc
chk = tuned.merge(STUDIES_DF[['study_tag', 'tuned_auc']],
                  left_on='experiment_name', right_on='study_tag',
                  how='left', suffixes=('_reg', '_bp'))
chk['drift'] = (chk['auc'] - chk['tuned_auc']).abs()
bad = chk[chk['drift'] > 0.01]
print(f"\n  AUC drift (registry pooled vs best_params) > 0.01: {len(bad)} studies")
if len(bad) > 0:
    print(bad[['experiment_name', 'auc', 'tuned_auc', 'drift']].to_string(index=False))

# per-city presence (tuned side)
PCDF = ResultRegistry.per_city_to_dataframe(experiments)
nb11_pc = PCDF[PCDF['notebook'] == REGISTRY_NOTEBOOK]
print(f"  NB11_V2 per-city rows (tuned): {len(nb11_pc)}")

# pooled vs mean(folds) gap from fold metrics
gap = tuned[['experiment_name', 'auc', 'auc_mean']].copy()
gap['gap'] = gap['auc'] - gap['auc_mean']
print(f"\n  pooled vs mean(folds) gap: mean={gap['gap'].mean():.4f} "
      f"min={gap['gap'].min():.4f} max={gap['gap'].max():.4f}")

# headline
show = tuned[['experiment_name', 'parquet_name', 'classifier_name', 'auc', 'f1', 'auc_mean']]
show = show.sort_values('auc', ascending=False)
print("\n  Tuned ranked by pooled AUC (top 12):")
print(show.head(12).to_string(index=False))

print("\n  DONE. NB13d and NB13e now read NB11_V2 from the registry via load_all().")
print("  (NB13d CELL 2b backfill auto-skips once NB11_V2 is present.)")


CELL M2: VERIFY
  Loaded: NB11_V2_experiments_20260703_083441.json (62 experiments, notebook=NB11_V2)
  Loaded: NB09b_experiments_20260628_003554.json (40 experiments, notebook=NB09b)
  Loaded: NB09c_v2_experiments_20260614_180054.json (74 experiments, notebook=NB09c_v2)
  Loaded: NB09d_experiments_20260621_232712.json (14 experiments, notebook=NB09d)
  Loaded: NB10a_v1_experiments_20260507_234428.json (6 experiments, notebook=NB10a_v1)
  Loaded: NB09aV3_v1_experiments_20260506_142934.json (47 experiments, notebook=NB09aV3_v1)
  Loaded: NB09aV5_v1_experiments_20260503_220957.json (47 experiments, notebook=NB09aV5_v1)
  Loaded: NB09aV4_v1_experiments_20260503_180614.json (47 experiments, notebook=NB09aV4_v1)
  Loaded: NB09a_v2_experiments_20260501_164920.json (44 experiments, notebook=NB09a_v2)
  Loaded: NB09eV3_v1_experiments_20260429_174010.json (18 experiments, notebook=NB09eV3_v1)
  Loaded: NB09cV3_v1_experiments_20260429_090246.json (48 experiments, notebook=NB09cV3_v1)
  Loaded: N